# Step 2: Forward Pass

**What is a "forward pass"?** Pushing one (or many) images through the network, layer by layer, to get a prediction out the other end. "Forward" because information flows in one direction: input -> hidden layer -> output. This step builds that mechanism and tests it with **random, untrained weights** - on purpose. Before training can make any sense, the untrained mechanics need to actually work: right shapes in, right shapes out, no errors.

All the code lives in [`network.py`](network.py) so Steps 3 and 4 can reuse it - this notebook explains what each piece does and why.

In [1]:
import numpy as np

from mnist_loader import load_mnist
from network import init_params, forward, relu, softmax

X_train, y_train, X_test, y_test = load_mnist()

## Starting weights: small and random, not zero

`init_params()` fills the weight matrices with small random numbers instead of zeros. **Why not start at zero?** If every weight started identical, every hidden neuron would receive the exact same input signal and compute the exact same output - they'd all be indistinguishable copies of each other, forever (training would update them all identically too). Random starting values break that symmetry, so each of the 128 hidden neurons can end up learning something different.

In [2]:
W1, b1, W2, b2 = init_params()
print("W1 (input -> hidden):", W1.shape)
print("b1:", b1.shape)
print("W2 (hidden -> output):", W2.shape)
print("b2:", b2.shape)
print()
print("A few actual starting weight values (W1):", np.round(W1[0, :5], 4))

W1 (input -> hidden): (784, 128)
b1: (128,)
W2 (hidden -> output): (128, 10)
b2: (10,)

A few actual starting weight values (W1): [ 0.0154 -0.0525  0.0379  0.0475 -0.0985]


## The two activation functions

- **ReLU** (hidden layer): `max(0, z)`. If the weighted sum is negative, the neuron outputs 0 ("not activated"); if positive, it passes the value through unchanged. Dead simple, but crucially *nonlinear* - without some nonlinearity here, stacking multiple layers would mathematically collapse into the same thing as one layer, and the network couldn't learn anything a single layer couldn't.
- **Softmax** (output layer): turns 10 raw scores into 10 probabilities that add up to 1. That makes the output directly interpretable as "how confident is the network in each digit", and is the standard choice for a classification problem with more than 2 classes.

In [3]:
example_scores = np.array([[2.0, -1.0, 0.5]])
print("Raw scores:", example_scores)
print("After ReLU:", relu(example_scores))
print("After softmax:", np.round(softmax(example_scores), 3), "(sums to", softmax(example_scores).sum(), ")")

Raw scores: [[ 2.  -1.   0.5]]
After ReLU: [[2.  0.  0.5]]
After softmax: [[0.786 0.039 0.175]] (sums to 1.0 )


## Running one image through the network

In [4]:
single_image = X_train[0:1]  # shape (1, 784) - forward() expects a batch, even of size 1
probs, cache = forward(single_image, W1, b1, W2, b2)

print("Output shape:", probs.shape)
print("Probabilities:", np.round(probs[0], 3))
print("Sum of probabilities:", probs.sum())
print()
print("Network's predicted digit:", probs.argmax())
print("Actual label:", y_train[0])

Output shape: (1, 10)
Probabilities: [0.073 0.073 0.116 0.127 0.183 0.035 0.126 0.121 0.018 0.128]
Sum of probabilities: 1.0

Network's predicted digit: 4
Actual label: 5


The network makes a prediction, and the probabilities correctly sum to 1 - the mechanics work. But the prediction is very likely wrong, and that's expected: **every weight is still random**. The network hasn't seen a single training example yet.

## Sanity check: an untrained network should guess at chance level

With 10 possible digits and zero training, the network shouldn't be able to do better than random guessing - about 10% accuracy. If it scored much higher than that right now, something in the code would be suspiciously wrong (e.g. accidentally leaking the answer in); if it scored much lower, something would be broken.

In [5]:
batch = X_train[:1000]
probs_batch, _ = forward(batch, W1, b1, W2, b2)
predictions = probs_batch.argmax(axis=1)

accuracy = (predictions == y_train[:1000]).mean()
print(f"Untrained accuracy on 1,000 samples: {accuracy*100:.1f}%")

Untrained accuracy on 1,000 samples: 9.2%


~10% - right where an untrained, randomly-guessing network should land. That's not a disappointing result, it's a **passing sanity check**: the forward pass mechanics work correctly, and there's nowhere to go from here but up, once training (Steps 3-4) starts actually adjusting these weights based on real feedback.

## Summary

- Weights start small and random (not zero) so different neurons can learn different things.
- ReLU (hidden layer) and softmax (output layer) are the two activation functions used here.
- The forward pass mechanics are verified correct: right output shape, probabilities sum to 1, and an untrained network scores right around the 10% chance level expected for 10 balanced classes.

Next: Step 3 measures exactly *how wrong* each prediction is (loss), and works out how to adjust every weight to make it a little less wrong (backpropagation).